In [0]:
from pyspark.sql.functions import *

raw_df = spark.range(0, 100).withColumn("order_id", col("id") + 5000) \
    .withColumn("product", element_at(array(lit("Phone"), lit("Tablet"), lit("Laptop")), (rand()*3 + 1).cast("int"))) \
    .withColumn("amount", (rand() * 1000).cast("decimal(10,2)")) \
    .withColumn("ingestion_timestamp", current_timestamp())

raw_df.write.format("delta").mode("overwrite").saveAsTable("orders_bronze")

display(spark.table("orders_bronze"))

In [0]:
from pyspark.sql.functions import upper, col

bronze_df = spark.table("orders_bronze")

silver_df = bronze_df.select(
    "order_id",
    upper(col("product")).alias("product_name"),
    col("amount"),
    col("ingestion_timestamp")
).filter(col("amount") > 500)

silver_df.write.format("delta").mode("overwrite").saveAsTable("orders_silver")

print("✨ Silver Layer is Ready! High-value orders filtered.")
display(spark.table("orders_silver"))

In [0]:

silver_table = spark.table("orders_silver")

gold_df = silver_table.groupBy("product_name").agg(
    sum("amount").alias("total_revenue"),
    count("order_id").alias("number_of_orders"),
    round(avg("amount"), 2).alias("avg_order_value")
).orderBy(col("total_revenue").desc())

gold_df.write.format("delta").mode("overwrite").saveAsTable("sales_summary_gold")

print("🏆 GOLD LAYER COMPLETE: Business Insights Generated!")
display(spark.table("sales_summary_gold"))

# Final Comment